### Part 2
Analyzing Top Mean-REs Detections

In [ ]:
from helperfunctions import helper as hfn
from helperfunctions import intern_constants as ic
from helperfunctions.pretty_print import PrettyPrint as pp
from helperfunctions.preprocessing import PreprocessingStep5
from helperfunctions.detection import Part2 as p2
import pandas as pd
from pathlib import Path
import numpy as np
import os
from glob import glob

# %matplotlib widget

In [ ]:
cfg_part2 = hfn.TrainConfig(config_name="part2", choose_val_set=2)

In [ ]:
print(f"test start:{cfg_part2.test_start_time}\n"
      f"test end:  {cfg_part2.test_end_time}")

In [ ]:
theta = p2.load_threshold_table()

In [ ]:
test_eval_file = glob(os.path.join(ic.PATH_PART2_EVAL_TEST, "*.csv")) [0]
test_eval_df = pd.read_csv(test_eval_file)

display(test_eval_df.head())

In [ ]:
files = glob(os.path.join(ic.PATH_PART2_DETECTIONS, "part2_dets_gdays_1.csv"))[0]


In [ ]:
selected_detections = pd.read_csv(files)
selected_detections = selected_detections.sort_values(by=[ic.MEAN_LOSS_PER_SAMPLE], ascending=False)
display(selected_detections)

In [ ]:

windows_df = p2.build_detection_windows(det_df=selected_detections, window_days=7)

display(windows_df.head(10))

selected_detections[ic.TS_COL] = pd.to_datetime(selected_detections[ic.TS_COL])
windows_df[ic.TS_COL] = pd.to_datetime(windows_df[ic.TS_COL])
windows_df["window_start"] = pd.to_datetime(windows_df["window_start"])
windows_df["window_end"] = pd.to_datetime(windows_df["window_end"])

windows_df = selected_detections.merge(
    windows_df,
    on=[ic.WT_ID, ic.TS_COL],
    how="left",
    validate="one_to_one"
)
display(windows_df.head())

In [ ]:

wt = 12
date = "2022-01-05 20:00:00"
wt_windows = windows_df[(windows_df[ic.WT_ID]== wt) & (windows_df[ic.TS_COL] == date)]
anomaly_spans_wt = [(row["window_start"], row["window_end"]) for _,row in wt_windows.iterrows()]
anomaly_spans_wt = sorted(anomaly_spans_wt, key=lambda span: span[0])
min_ts = anomaly_spans_wt[0][0]
max_ts = anomaly_spans_wt[-1][1]

In [ ]:
pp.print_loss(test_eval_df,
              dpi=300,
              y_limits=((0,0.2),(0.2,100)),
              title="Test Set: Mean Turbine Loss per Sample",
              wt_id= [wt],
              ts_range=(min_ts, max_ts),
              anomaly_spans=anomaly_spans_wt,
              #mark_threshold=theta,
              anom_span_label="Detection Windows",
              detection_ts=[date],
              figsize=(10,6),
              save_filename="p2_top_re_mean_loss_1.png",
              impute_label="Data Imputation"
              )

### Filter Top N Signals with highest RE

In [ ]:
top_signals_wt = p2.get_top_sigs_for_detections(
    eval_df=p2.drop_imputations(test_eval_df),
    selected_detections=selected_detections[(selected_detections[ic.WT_ID]==wt ) & (selected_detections[ic.TS_COL]==date)],
    wt_id=wt,
    top_n_signals=3
)

print(top_signals_wt)

In [ ]:
cols = [ic.MEAN_LOSS_PER_SAMPLE]+top_signals_wt[0][1]
print(cols)
sd_filter = selected_detections[[ic.TS_COL, ic.WT_ID]+ cols]
sd_filter[cols] = np.floor(sd_filter[cols]*100)/100
sd_filter = sd_filter.iloc[[0]]
sd_filter = sd_filter.rename(columns={ic.MEAN_LOSS_PER_SAMPLE: "MSE Loss"})
ltx = sd_filter.to_latex(index=False, escape=True, float_format="%.2f")
with open(ic.PATH_PRINTS/"p2_1_wt12_loss.tex", "w", encoding="utf-8") as f:
    f.write(ltx)

display(sd_filter)

In [ ]:
print(len(anomaly_spans_wt))
print(anomaly_spans_wt)
print(len(top_signals_wt))
print(top_signals_wt)

In [ ]:
for idx,(ts,sig_list) in enumerate(top_signals_wt):
    for sig in sig_list:
        pp.print_loss(test_eval_df,
                    dpi=300,
                    y_limits=((0,0.22),(0.22,800)),
                    title=f"Test Set: WT {wt} {sig}",
                    wt_id= [wt],
                    values=sig,
                    ts_range=(min_ts, max_ts),
                    anomaly_spans=[anomaly_spans_wt[idx]],
                    mark_threshold=None,
                    anom_span_label="Detection Windows on Top N REs",
                    detection_ts=[ts],
                    detection_label="Top-RE Detection",
                    figsize=(10,6),
                    save_filename=f"p2_top_signals_1_{sig}.png",
                    impute_label="Data Imputation"
                    )

In [ ]:
top_sigs_no_prefix_wt = [(ts, sig[3:]) for sig in sig_list  for (ts,sig_list) in top_signals_wt]
print(top_sigs_no_prefix_wt)

In [ ]:
for (ts,sig) in top_sigs_no_prefix_wt:
    print(sig[:-5])

In [ ]:
fp = Path(ic.PATH_IMPUTED).glob(f"WT_ID_{wt}*")
wt_df = pd.read_csv(next(iter(fp)))

for (ts,sig) in top_sigs_no_prefix_wt:
    pp.print_loss(wt_df, 
                  values=sig,
                  title=f"WT {wt}: Raw Signal:{sig} ",
                  wt_id=[wt],
                  ts_range= (min_ts, max_ts),
                  anomaly_spans=anomaly_spans_wt,
                  detection_label="Detected Anomaly",
                  detection_ts= [ts],
                  y_label = sig[-4:],
                  figsize=(10,6),
                  save_filename=f"p2_top_re_raw_signal_1_{sig[:-5]}")



### Power Curve 

In [ ]:
pre_step5 = PreprocessingStep5()
df_pc = pre_step5._prepare_power_curve()
df_pc.head()

In [ ]:
test_start = cfg_part2.test_start_time
test_end = cfg_part2.test_end_time

wind_col = "Wind speed (m/s)"
power_col = "Power (kW)"

imputed_files = sorted(Path(ic.PATH_IMPUTED).glob("*.csv"))

wt_farm : list[pd.DataFrame] = []


wt_farm_df = pd.read_csv(ic.PATH_PART2_WT_FARM /  "wt_farm_pc_df.csv")
wt_farm_df = wt_farm_df.sort_values([ic.WT_ID, ic.TS_COL])
wt_farm_df.head()
wt_farm_df[ic.TS_COL] = pd.to_datetime(wt_farm_df[ic.TS_COL])

In [ ]:
df_det_keys = selected_detections[[ic.WT_ID, ic.TS_COL]].copy()

df_det_points = df_det_keys.merge(
    wt_farm_df[[ic.WT_ID,ic.TS_COL, wind_col, power_col]],
    on=[ic.WT_ID, ic.TS_COL],
    how="left",
    validate="one_to_one"
)

df_det_points.head()

In [ ]:
pp.print_powercurve(df_pc=df_pc,
                    df_wts=p2.drop_imputations( wt_farm_df, pc=False),
                    df_detections=df_det_points[(df_det_points[ic.WT_ID]== wt) & (df_det_points[ic.TS_COL]==date )],
                    detections_label=f"Detection on WT {wt}",
                    highlight_wt=wt,
                    highlight_ts_range=(pd.to_datetime(date)-pd.Timedelta(hours=2), pd.to_datetime(date)+pd.Timedelta(hours=2)),
                    alpha_det=0.7,
                    fig_size=(9,4),
                    save_dir=Path(ic.PATH_PRINTS),
                    file_name="p2_top_re_1_pc.png"
                    )

In [ ]:
pp.print_loss(wt_df, 
                  values=wind_col,
                  title=f"WT {wt}: Raw Signal:{wind_col} ",
                  ts_range= (min_ts, max_ts),
                  anomaly_spans=anomaly_spans_wt,
                  detection_label="Detected Anomaly",
                  detection_ts= [ts],
                  y_label = wind_col[-4:-1],
                  anom_span_label="Anomaly Window")

In [ ]:
pp.print_loss(wt_df, 
                  values=power_col,
                  title=f"WT {wt}: Raw Signal:{power_col} ",
                  ts_range= (min_ts, max_ts),
                  anomaly_spans=anomaly_spans_wt,
                  detection_label="Detected Anomaly",
                  detection_ts= [ts],
                  y_label = power_col[-3:-1],
                  anom_span_label="Anomaly Window",
                  save_filename="p2_1_raw_power.png")

In [ ]:
pp.plot_wind_vs_nacelle(df=wt_df,
                        ts_mark=pd.to_datetime(ts),
                        ts_range=(ts-pd.Timedelta(days=5), ts+pd.Timedelta(days=5)),
                        title="Wind Direction vs Nacelle Pos.",
                        save_filename="p2_1_wind_dir_nacelle_pos.png")

In [ ]:
status_logs = pd.read_csv(ic.PATH_PENMANSHIEL / "Status_Logs_Complete_ID_12.csv")
status_logs["Timestamp start"] = pd.to_datetime(status_logs["Timestamp start"], errors="coerce")
status_logs["Timestamp end"] = pd.to_datetime(status_logs["Timestamp end"], errors="coerce")

In [ ]:
display(status_logs.head())

In [ ]:
detection_ts = pd.to_datetime(date)

In [ ]:
filter = status_logs[( detection_ts >= status_logs["Timestamp start"] ) & (detection_ts <= status_logs["Timestamp end"] )]

display(filter)

In [ ]:
delta = pd.Timedelta(hours=3)
det_start = detection_ts - delta
det_end = detection_ts + delta

In [ ]:
filter = status_logs[(status_logs["Timestamp start"] <= det_end) & (status_logs["Timestamp end"] >= det_start)]
display(filter)
print(detection_ts)

In [ ]:
ltx = filter.to_latex(index=False, escape=True)
with open(ic.PATH_PRINTS/"p2_1_status_logs.tex", "w", encoding="utf-8") as f:
    f.write(ltx)